# Statistical Testing

In [1]:
import os
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel, wilcoxon, shapiro

DATASETS = ['FD001', 'FD002']
MODELS = {
    'Baseline LSTM': 'lstm_baseline',
    'Improved LSTM': 'lstm_improved',
    '1D-CNN': 'cnn',
    'Hybrid CNN-LSTM': 'cnn_lstm'
}

metrics = ['rmse', 'mae', 'nasa']
alpha = 0.05

for ds in DATASETS:
    print(f"\n{'='*70}")
    print(f"STATISTICAL TESTS – {ds}")
    print(f"{'='*70}")
    
    # Load all model metrics for this dataset
    data = {}
    for name, prefix in MODELS.items():
        path = f'../results/metrics/{prefix}_{ds}_metrics.csv'
        if os.path.exists(path):
            df = pd.read_csv(path)
            data[name] = df
            print(f"Loaded {name}: {len(df)} seeds")
        else:
            print(f"WARNING: Missing {path}")
    
    if len(data) < 2:
        continue
    
    model_names = list(data.keys())
    
    for metric in metrics:
        print(f"\n--- Metric: {metric.upper()} ---")
        results = []
        
        for i in range(len(model_names)):
            for j in range(i+1, len(model_names)):
                a, b = model_names[i], model_names[j]
                vals_a = data[a][metric].values
                vals_b = data[b][metric].values
                
                # Shapiro-Wilk on the paired differences
                diff = vals_a - vals_b
                _, p_shapiro = shapiro(diff)
                
                # Paired tests
                _, p_ttest = ttest_rel(vals_a, vals_b)
                try:
                    _, p_wilcox = wilcoxon(vals_a, vals_b)
                except ValueError:
                    p_wilcox = np.nan
                
                sig = (p_ttest < alpha) or (p_wilcox < alpha)
                
                results.append({
                    'Comparison': f"{a} vs {b}",
                    'Shapiro p': round(p_shapiro, 4),
                    'Paired t p': f"{p_ttest:.2e}",
                    'Wilcoxon p': f"{p_wilcox:.2e}",
                    'Significant (α=0.05)': sig
                })
        
        res_df = pd.DataFrame(results)
        print(res_df.to_string(index=False))
        res_df.to_csv(f'../results/metrics/stats_{metric}_{ds}.csv', index=False)


STATISTICAL TESTS – FD001
Loaded Baseline LSTM: 10 seeds
Loaded Improved LSTM: 10 seeds
Loaded 1D-CNN: 10 seeds
Loaded Hybrid CNN-LSTM: 10 seeds

--- Metric: RMSE ---
                      Comparison  Shapiro p Paired t p Wilcoxon p  Significant (α=0.05)
  Baseline LSTM vs Improved LSTM     0.8015   6.91e-01   8.46e-01                 False
         Baseline LSTM vs 1D-CNN     0.3051   3.34e-06   1.95e-03                  True
Baseline LSTM vs Hybrid CNN-LSTM     0.7976   4.82e-03   9.77e-03                  True
         Improved LSTM vs 1D-CNN     0.2016   1.93e-06   1.95e-03                  True
Improved LSTM vs Hybrid CNN-LSTM     0.3817   3.70e-03   1.37e-02                  True
       1D-CNN vs Hybrid CNN-LSTM     0.4119   1.97e-04   1.95e-03                  True

--- Metric: MAE ---
                      Comparison  Shapiro p Paired t p Wilcoxon p  Significant (α=0.05)
  Baseline LSTM vs Improved LSTM     0.2304   9.74e-01   7.70e-01                 False
         Baseline L